In [1]:
# ============================================================
# 1_data_ingestion.ipynb
# NYC TLC Yellow Taxi Data Ingestion & Cleaning
# ============================================================

# Install Spark (Colab)
!pip install pyspark==3.5.1 -q

import os
import subprocess
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# -----------------------------
# Create folders
# -----------------------------
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

# -----------------------------
# Start Spark (Colab-safe config)
# -----------------------------
spark = (
    SparkSession.builder
    .appName("TLC_Data_Ingestion")
    .config("spark.driver.memory", "6g")
    .config("spark.sql.shuffle.partitions", "100")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

# -----------------------------
# Download 2019 Yellow Taxi Data (~1GB+)
# -----------------------------
base_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/"

for month in range(1, 13):
    filename = f"yellow_tripdata_2019-{month:02d}.parquet"
    filepath = f"data/raw/{filename}"

    if not os.path.exists(filepath):
        print(f"Downloading {filename}...")
        subprocess.run(["wget", "-q", "-P", "data/raw/", base_url + filename])

# -----------------------------
# Load Dataset
# -----------------------------
df_raw = spark.read.option("mergeSchema", "true") \
    .parquet("data/raw/yellow_tripdata_2019-*.parquet")

print("Total Columns:", len(df_raw.columns))

# -----------------------------
# Basic Cleaning + Minimal Feature Engineering
# -----------------------------
df = df_raw

# Trip duration (seconds)
df = df.withColumn(
    "trip_seconds",
    F.unix_timestamp("tpep_dropoff_datetime") -
    F.unix_timestamp("tpep_pickup_datetime")
)

# Extract pickup date for partitioning
df = df.withColumn(
    "pickup_date",
    F.to_date("tpep_pickup_datetime")
)

# Apply cleaning filters
df_clean = df.filter(
    (F.col("trip_seconds") > 0) &
    (F.col("trip_seconds") < 21600) &      # < 6 hours
    (F.col("trip_distance") > 0) &
    (F.col("trip_distance") < 200) &
    (F.col("fare_amount") > 0) &
    (F.col("total_amount") > 0)
)

# -----------------------------
# Write Partitioned Parquet (Required by Brief)
# -----------------------------
output_path = "data/processed/tlc_cleaned_parquet"

df_clean.repartition(100, "pickup_date") \
    .write \
    .mode("overwrite") \
    .partitionBy("pickup_date") \
    .parquet(output_path)

print("Cleaned dataset successfully saved to:", output_path)

# -----------------------------
# Stop Spark
# -----------------------------
spark.stop()

print("1_data_ingestion.ipynb completed successfully.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 1.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 19.6 MB/s eta 0:00:00
Total Columns: 19
Cleaned dataset successfully saved to: data/processed/tlc_cleaned_parquet
1_data_ingestion.ipynb completed successfully.


In [2]:
import shutil
from google.colab import files

folder_to_download = "/content/data/processed/tlc_cleaned_parquet"
zip_filename = "tlc_cleaned_parquet.zip"

print(f"Zipping folder: {folder_to_download} to {zip_filename}...")
shutil.make_archive(zip_filename.split('.')[0], 'zip', folder_to_download)

print(f"Downloading {zip_filename}...")
files.download(zip_filename)
print("Download initiated. Please check your browser's downloads.")

Zipping folder: /content/data/processed/tlc_cleaned_parquet to tlc_cleaned_parquet.zip...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download initiated. Please check your browser's downloads.
